# Exploring Datasets for thesis


In [ ]:
import pandas as pd
import re
import pycountry
import matplotlib.pyplot as plt

### UN Food and Agriculture Fisheries Tuna Catch Dataset (1918-2023)

Data from UN FAO (https://zenodo.org/records/17494424)
Metadata/explanation here: https://www.fao.org/fishery/geonetwork/srv/eng/catalog.search#/metadata/global_nominal_catch_firms_level0

Need:

- codes for:
  - Fishing Fleet
    - uses ISO 3166 country codes
  - Species
    - BET Bigeye Tuna
    - YFT Yellowfin Tuna
    - BFT Atlantic Bluefin Tuna
    - PBF Pacific Bluefin Tuna
    - SBF Southern Bluefin Tuna
  - Measurement Type
    - NC Nominal Catch
    - NL Nominal Landings
    - DL or DD is Discarded Alive or Dead
  - unit
    - t (metric tons)


In [ ]:
fao_all = pd.read_csv("datasets/global_nominal_catch_firms_level0_harmonized.csv")

#get year from the first four numbers in time_start
fao_all["year"] = fao_all["time_start"].str.extract(r"(\d{4})").astype(int)
fao_all.head()

In [ ]:
#atlantic bluefin tuna subset
atlantic_bluefin_tuna = fao_all[fao_all["species"].str.contains("BFT", case=False, na=False)]
atlantic_bluefin_tuna.head() #4000 rows

#column for year: take first four numbers in time_start column
atlantic_bluefin_tuna["year"] = atlantic_bluefin_tuna["time_start"].str.extract(r"(\d{4})").astype(int)
atlantic_bluefin_tuna.head()

#group by year, and for each source_authority, sum the catch
catch_by_year = atlantic_bluefin_tuna.groupby(["year", "source_authority"])["measurement_value"].sum().reset_index()
catch_by_year.head()


In [ ]:
#bar plot of measurement_value by year, with different colors for each source_authority
plt.figure(figsize=(12, 6))
for authority in catch_by_year["source_authority"].unique():
    authority_data = catch_by_year[catch_by_year["source_authority"] == authority]
    plt.bar(authority_data["year"], authority_data["measurement_value"], label=authority)
plt.xlabel("Year")
plt.ylabel("Total Catch (measurement_value)")
plt.title("Total Catch of Atlantic Bluefin Tuna by Year and Source Authority")
plt.legend()
plt.show()


In [ ]:
#subset for all tuna: BFT, YFT, PBF, SBF, BET
all_tuna = fao_all[fao_all["species"].str.contains("BFT|PBF|SBF|YFT|BET", case=False, na=False)]





#bar plot of measurement_value by year, with different colors for each species
catch_by_year_tuna = all_tuna.groupby(["year", "species"])["measurement_value"].sum().reset_index()
plt.figure(figsize=(12, 6))
for species in catch_by_year_tuna["species"].unique():
    species_data = catch_by_year_tuna[catch_by_year_tuna["species"] == species].groupby("year")["measurement_value"].sum().reset_index()
    plt.bar(species_data["year"], species_data["measurement_value"], label=species)
plt.xlabel("Year")
plt.ylabel("Total Catch (measurement_value)")
plt.title("Total Catch of Tuna by Year and Species")
plt.legend()

#same plot but exclude YFT
catch_by_year_tuna_no_yft = all_tuna[~all_tuna["species"].str.contains("YFT", case=False, na=False)].groupby(["year", "species"])["measurement_value"].sum().reset_index()
plt.figure(figsize=(12, 6))
for species in catch_by_year_tuna_no_yft["species"].unique():
    species_data = catch_by_year_tuna_no_yft[catch_by_year_tuna_no_yft["species"] == species].groupby("year")["measurement_value"].sum().reset_index()
    plt.bar(species_data["year"], species_data["measurement_value"], label=species)
plt.xlabel("Year")
plt.ylabel("Total Catch (measurement_value)")
plt.title("Total Catch of Tuna (excluding YFT) by Year and Species")
plt.legend()
plt.show()

In [ ]:
#names? 

unrecognized = fao_all[fao_all["fishing_fleet"].notna()]["fishing_fleet"].apply(
    lambda x: x if pycountry.countries.get(alpha_3=x) is None else None
).dropna().unique()
print(unrecognized)

In [ ]:
fao_all["country_name"] = fao_all["fishing_fleet"].apply(
    lambda x: country.name if pd.notnull(x) and (country := pycountry.countries.get(alpha_3=x)) else None
)

fao_all.head()

## NOAA Import/Export Dataset cleaning for prototype Sankey diagram

Downloaded from https://www.fisheries.noaa.gov/foss/f?p=215:2:14285563639484:::::
Import data from 1972 to 2026. Source file doesn't have all fish products, only ones that are relevant to sushi or Japanese food.

In [30]:
import_all = pd.read_csv("datasets/ANNUAL TRADE-NO AGGREGATION_.csv")
first_row = import_all.iloc[0]
#rename columns using first row
import_all.columns = first_row
import_all = import_all[1:] #drop first row


import_all.head()

/var/folders/s6/l4_0rpsd3xs3vxs6x047x7t80000gn/T/ipykernel_18177/2239496738.py:1: DtypeWarning: Columns (0,6,7,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  import_all = pd.read_csv("datasets/ANNUAL TRADE-NO AGGREGATION_.csv")


,Year,Continent,Country Name,Product Name,Edible code,US Customs District,Census Country Code,Census District Code,FAO Country Code,HTS Number,Value (USD),Volume (kg),Calculated Duty (USD)
1,2026,NORTH AMERICA,CANADA,SALMON CHINOOK FRESH FARMED,E,"BUFFALO, NY",1220,09,124,0302130013,"29,021","1,836",0
2,2026,NORTH AMERICA,CANADA,SALMON CHINOOK FRESH FARMED,E,"SEATTLE, WA",1220,30,124,0302130013,"2,145,067","112,804",0
3,2026,OCEANIA,NEW ZEALAND,SALMON CHINOOK FRESH FARMED,E,"NEW YORK, NY",6141,10,554,0302130013,"592,978","36,853","88,946"
4,2026,OCEANIA,NEW ZEALAND,SALMON CHINOOK FRESH FARMED,E,"LOS ANGELES, CA",6141,27,554,0302130013,"1,963,164","128,328","294,477"
5,2026,OCEANIA,NEW ZEALAND,SALMON CHINOOK FRESH FARMED,E,"SAN FRANCISCO, CA",6141,28,554,0302130013,"115,265","7,184","17,289"


In [43]:
#Look at only Bluefin Tuna, and simplify dataframe for export and use in Sankey diagram



import_tuna = import_all[import_all["Product Name"].str.contains("TUNA", case=False, na=False)]
#drop columns that aren't used
import_tuna = import_tuna[["Year","Continent", "Country Name", "Product Name", "US Customs District", "Value (USD)", "Volume (kg)"]]

#subset for only bluefin tuna
import_bluefin_tuna = import_tuna[import_tuna["Product Name"].str.contains("BLUEFIN", case=False, na=False)]
#year, Value USD and Volume kg should be int or float, not string.
import_bluefin_tuna["Year"] = import_bluefin_tuna["Year"].astype(int)
# value and volume have commas, so remove commas and convert to int
import_bluefin_tuna["Value (USD)"] = import_bluefin_tuna["Value (USD)"].str.replace(",", "").astype(int)
import_bluefin_tuna["Volume (kg)"] = import_bluefin_tuna["Volume (kg)"].str.replace(",", "").astype(int)

import_bluefin_tuna.head()


/var/folders/s6/l4_0rpsd3xs3vxs6x047x7t80000gn/T/ipykernel_18177/3347722567.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  import_bluefin_tuna["Year"] = import_bluefin_tuna["Year"].astype(int)
/var/folders/s6/l4_0rpsd3xs3vxs6x047x7t80000gn/T/ipykernel_18177/3347722567.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  import_bluefin_tuna["Value (USD)"] = import_bluefin_tuna["Value (USD)"].str.replace(",", "").astype(int)
/var/folders/s6/l4_0rpsd3xs3vxs6x047x7t80000gn/T/ipykernel_18177/3347722567.py:

,Year,Continent,Country Name,Product Name,US Customs District,Value (USD),Volume (kg)
207,2026,ASIA,JAPAN,"TUNA BLUEFIN ATLANTIC,PACIFIC FRESH","NEW YORK, NY",35213,1272
208,2026,ASIA,JAPAN,"TUNA BLUEFIN ATLANTIC,PACIFIC FRESH","LOS ANGELES, CA",86467,3094
209,2026,ASIA,JAPAN,"TUNA BLUEFIN ATLANTIC,PACIFIC FRESH","HONOLULU, HI",39025,712
210,2026,ASIA,JAPAN,"TUNA BLUEFIN ATLANTIC,PACIFIC FRESH","MIAMI, FL",34912,741
211,2026,ASIA,JAPAN,"TUNA BLUEFIN ATLANTIC,PACIFIC FRESH","HOUSTON-GALVESTON, TX",35669,1073


In [49]:
#export to csv for use in sankey diagram
import_bluefin_tuna.to_csv("datasets/bluefin_tuna_imports.csv", index=False)